In [ ]:
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib
from pathlib import Path

root = Path('.')
base = pd.read_csv(root/'events_baseline.csv')
data = pd.read_csv(root/'events_raw.csv')
len(base), len(data)

## Build preprocessing + model pipeline
We one-hot encode categoricals and scale numeric fields. Then IsolationForest learns what "normal" looks like.

In [ ]:
numeric = ['bytes_transferred','hour_of_day','ip_reputation_score','session_length_sec']
categorical = ['role','resource_type','action','geo_region']
ct = ColumnTransformer([
    ('num', StandardScaler(), numeric),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical)
])
iso = IsolationForest(n_estimators=200, contamination=0.03, random_state=42)
pipe = Pipeline([('prep', ct), ('iso', iso)])
pipe.fit(base[numeric+categorical])
joblib.dump(pipe, root/'iforest_model.joblib')
print('Model saved to iforest_model.joblib')

## Score the full dataset
We compute an anomaly score per row. Lower scores indicate more anomalous rows in IsolationForest.

In [ ]:
pipe = joblib.load(root/'iforest_model.joblib')
X = data[numeric+categorical]
scores = pipe['iso'].score_samples(pipe['prep'].transform(X))
data['iso_score'] = scores
data.head(3)

## Save scored data for the next step
We write a CSV with the new `iso_score` column so we can select a threshold in the next lab step.

In [ ]:
data.to_csv(root/'events_scored.csv', index=False)
print('Saved events_scored.csv with iso_score column')